# Data Backup 

In [2]:
import os
import glob
import pickle
import sys
import re
# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)

import pandas as pd
import datetime as dt
from datetime import date, datetime
import numpy as np

from server_config import datapath, raw_path

today = date.today().strftime("%d%m%Y")
today_day = pd.to_datetime('today').normalize()

### Backup -> 2023-11-27

In [ ]:

# Define the pattern for big backup passive data files
file_pattern_back_1 = os.path.join(raw_path, 'tiki_backup_files/tiki_backup_*.csv')

# Use glob to find all matching files
big_backup_files = glob.glob(file_pattern_back_1)

# Define the dtype for columns that are known to be problematic
dtype_spec = {
    'startTimestamp': 'str',  # Load as string initially
    'endTimestamp': 'str'     # Load as string initially
}

# Create a list to hold all the dataframes
all_dfs = []

# Loop over the files and read them with date parsing
for filename in big_backup_files:
    df = pd.read_csv(
        filename,
        dtype=dtype_spec,  # Load timestamps as strings first
        low_memory=False  # Ensure proper memory handling
    )
    
    # Convert the timestamp columns to datetime, ensuring proper parsing of ISO 8601 format
    df['starttimestamp'] = pd.to_datetime(df['startTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')
    df['endtimestamp'] = pd.to_datetime(df['endTimestamp'], format='%Y-%m-%dT%H:%M:%S%z', errors='coerce')

    
    # Additional processing (e.g., adding date and index columns)
    match = re.match(r'tiki_backup_(\d{4}-\d{2}-\d{2})_(\d+)\.csv', os.path.basename(filename))
    if match:
        date_part = match.group(1)  # Extract the date
        index_part = int(match.group(2))  # Extract the index
        # Add the date and index as new columns to the dataframe
        df['date'] = date_part
        df['index'] = index_part

    all_dfs.append(df)

# Concatenate all dataframes
df_backup_big = pd.concat(all_dfs, ignore_index=True)

# Sort the merged dataframe by 'date' and 'index'

# Optionally, drop the 'date' and 'index' columns if they are no longer needed
df_backup_big.drop(columns=['date', 'index'], inplace=True)


In [ ]:
# Convert the 'startTimestamp' and 'endTimestamp' columns to datetime objects
df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')
# Convert the 'startTimestamp' and 'endTimestamp' columns to datetime objects
df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')

# Adjust for timezone offset
df_backup_big['startTimestamp'] = df_backup_big['startTimestamp'] + pd.to_timedelta(df_backup_big['timezoneOffset'], unit='m')
df_backup_big['endTimestamp'] = df_backup_big['endTimestamp'] + pd.to_timedelta(df_backup_big['timezoneOffset'], unit='m')


# Format the datetime objects to the desired format
df_backup_big['startTimestamp'] = df_backup_big['startTimestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_backup_big['endTimestamp'] = df_backup_big['endTimestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

df_backup_big['startTimestamp'] = pd.to_datetime(df_backup_big['startTimestamp'], errors='coerce')
df_backup_big['endTimestamp'] = pd.to_datetime(df_backup_big['endTimestamp'], errors='coerce')

### Backup 21.11.2023 - 21.05.2024

In [3]:
# small backup passive data
#file_pattern_back_1 = os.path.join(datapath, 'raw/tiki_backup_files/export_tiki_27052024/"epoch_part*.csv"')
datapath_back = os.path.join(raw_path, "tiki_backup_files/export_tiki_21052024/")
file_pattern_back_2 = os.path.join(datapath_back,"epoch_part*.csv")  # Adjust the path and extension if needed

backup_files = glob.glob(file_pattern_back_2)
file_list = glob.glob(file_pattern_back_2)
file_list.sort()
df_backup_small = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [4]:
df_backup_small["start_end"] = df_backup_small["endTimestamp"] - df_backup_small["startTimestamp"]
df_backup_small["startTimestamp"] = pd.to_datetime(df_backup_small["startTimestamp"],unit='ms')
df_backup_small["endTimestamp"] = pd.to_datetime(df_backup_small["endTimestamp"],unit='ms')

df_backup_small["customer"] = df_backup_small.customer.str.split("@").str.get(0)
df_backup_small["customer"] = df_backup_small["customer"].str[:4]

df_backup_small['startTimestamp'] = df_backup_small['startTimestamp'] + pd.to_timedelta(df_backup_small['timezoneOffset'], unit='m')
df_backup_small['endTimestamp'] = df_backup_small['endTimestamp'] + pd.to_timedelta(df_backup_small['timezoneOffset'], unit='m')

### Backup 20.05.2024 - 11.11.2024

In [3]:
datapath_back_3 = os.path.join(raw_path, "tiki_backup_files/export_tiki_11112024/")
file_pattern_back_3 = os.path.join(datapath_back_3,"epoch_part*.csv")  # Adjust the path and extension if needed

backup_files_3 = glob.glob(file_pattern_back_3)
file_list_3 = glob.glob(file_pattern_back_3)
file_list_3.sort()
df_backup_small_3 = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list_3), ignore_index=True)

In [5]:
df_backup_small_3.tail()

,customer,source,startTimestamp,endTimestamp,type,valueType,doubleValue,longValue,booleanValue,dateValue,stringValue,generation,trustworthiness,medicalGrade,userReliability,chronologicalExactness,timezoneOffset,createdAt
45161125,oN5YLegpMQTRBqDF,Nokia,1730989500000,1.730990e+12,Steps,10,7.0,NaN,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,60.0,1730991923139
45161126,oN5YLegpMQTRBqDF,Nokia,1730989517000,1.730990e+12,HeartRate,20,NaN,75.0,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,60.0,1730991923139
45161127,oN5YLegpMQTRBqDF,Nokia,1730990088000,1.730990e+12,HeartRate,20,NaN,86.0,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,60.0,1730991923140
45161128,oN5YLegpMQTRBqDF,Nokia,1730990707000,1.730991e+12,HeartRate,20,NaN,79.0,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,60.0,1730991923140
45161129,oN5YLegpMQTRBqDF,Nokia,1730991296000,1.730991e+12,HeartRate,20,NaN,77.0,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,60.0,1730991923140


In [8]:
df_backup_small_3["start_end"] = df_backup_small_3["endTimestamp"] - df_backup_small_3["startTimestamp"]
df_backup_small_3["startTimestamp"] = pd.to_datetime(df_backup_small_3["startTimestamp"],unit='ms')
df_backup_small_3["endTimestamp"] = pd.to_datetime(df_backup_small_3["endTimestamp"],unit='ms')

df_backup_small_3["customer"] = df_backup_small_3.customer.str.split("@").str.get(0)
df_backup_small_3["customer"] = df_backup_small_3["customer"].str[:4]

df_backup_small_3['startTimestamp'] = df_backup_small_3['startTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')
df_backup_small_3['endTimestamp'] = df_backup_small_3['endTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')

### Backup 11.11.2025 - 05.05.2025

In [ ]:
datapath_back_3 = os.path.join(raw_path, "tiki_backup_files/export_tiki_05052025/")
file_pattern_back_3 = os.path.join(datapath_back_3,"epoch_part*.csv")  # Adjust the path and extension if needed

backup_files_3 = glob.glob(file_pattern_back_3)
file_list_3 = glob.glob(file_pattern_back_3)
file_list_3.sort()
df_backup_small_3 = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list_3), ignore_index=True)

In [ ]:
df_backup_small_3["start_end"] = df_backup_small_3["endTimestamp"] - df_backup_small_3["startTimestamp"]
df_backup_small_3["startTimestamp"] = pd.to_datetime(df_backup_small_3["startTimestamp"],unit='ms')
df_backup_small_3["endTimestamp"] = pd.to_datetime(df_backup_small_3["endTimestamp"],unit='ms')

df_backup_small_3["customer"] = df_backup_small_3.customer.str.split("@").str.get(0)
df_backup_small_3["customer"] = df_backup_small_3["customer"].str[:4]

df_backup_small_3['startTimestamp'] = df_backup_small_3['startTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')
df_backup_small_3['endTimestamp'] = df_backup_small_3['endTimestamp'] + pd.to_timedelta(df_backup_small_3['timezoneOffset'], unit='m')

## Concat 4 dataframes

In [9]:
latest_timestamp_big = df_backup_big['startTimestamp'].max()
latest_timestamp_small = df_backup_small['startTimestamp'].max()

# Filter the second dataframe to include only entries after the latest timestamp
df_backup_small_filtered = df_backup_small[df_backup_small['startTimestamp'] > latest_timestamp_big]

In [10]:
df_backup_3_filtered = df_backup_small_3[df_backup_small_3['startTimestamp'] > latest_timestamp_small]

In [11]:
# 1️⃣  Sort once, not twice
frames_sorted = [
    df_backup_big.sort_values("startTimestamp"),
    df_backup_small_filtered.sort_values("startTimestamp"),
    df_backup_3_filtered.sort_values("startTimestamp"),
]

result_df_final = (
    pd.concat(frames_sorted, ignore_index=True, copy=False)   # copy=False where possible
        .sort_values("startTimestamp", ignore_index=True)     # final global sort
)


In [12]:
result_df_final["customer"] = result_df_final.customer.str.split("@").str.get(0)
result_df_final["customer"] = result_df_final["customer"].str[:4]

In [13]:
print("Minimum timestamp:", result_df_final['startTimestamp'].min())
print("Maximum timestamp:", result_df_final['startTimestamp'].max())


Minimum timestamp: 2023-03-27 04:46:00
Maximum timestamp: 2025-01-20 02:19:41


In [ ]:
result_df_final_merged.drop(columns=['valueType', 'createdAt', 'source', 'trustworthiness', 'medicalGrade', 'generation'], inplace=True)

In [ ]:
object_cols = ["booleanValue", "stringValue", "customer", "type"] 

In [ ]:
# Fill NaN values with -99 for the specified columns
for col in object_cols:
    result_df_final_merged[col] = result_df_final_merged[col].fillna(-99)

# Convert "booleanValue" to boolean
result_df_final_merged['booleanValue'] = result_df_final_merged['booleanValue'].apply(lambda x: bool(x) if x != -99 else False)

result_df_final_merged['stringValue'] = result_df_final_merged['stringValue'].astype('string')
result_df_final_merged['customer'] = result_df_final_merged['customer'].astype('string')
result_df_final_merged['type'] = result_df_final_merged['type'].astype('string')


In [ ]:
# Calculate memory usage in bytes
memory_usage_bytes = result_df_final.memory_usage(deep=True).sum()

# Convert to megabytes
memory_usage_mb = memory_usage_bytes / (1024 ** 2)

# Convert to gigabytes
memory_usage_gb = memory_usage_bytes / (1024 ** 3)

# Convert to terabytes
memory_usage_tb = memory_usage_bytes / (1024 ** 4)

print(f"Memory usage: {memory_usage_bytes} bytes")
print(f"Memory usage: {memory_usage_mb:.2f} MB")
print(f"Memory usage: {memory_usage_gb:.2f} GB")
print(f"Memory usage: {memory_usage_tb:.2f} TB")


In [ ]:
backup_path = preprocessed_path + "/backup_data_passive_general.feather"

In [ ]:
# Save to HDF5 format
result_df_final_merged.to_feather(backup_path)